
## Folder 05 — File 03: Data Type and Format Validation:


**Purpose**: Validate whether Silver records contain correctly typed and consistently formatted values.

This notebook checks:

1. Email address formats.
2. Phone number formats.
3. Date and timestamp validity.
4. Numeric data types.
5. Allowed categorical values.
6. Percentage and decimal ranges.
7. Currency codes and identifier formats.

**Important distinction:** A column having the correct Spark data type does not mean its values are valid. For example, a string column can contain an invalid email address.


## Important technical notes:


- Phone validation is intentionally basic. A number can match the pattern and still be unassigned or invalid. Real phone verification requires a proper numbering library or verification service.
- Currency validation does not establish that a currency is supported by your business. The three-letter pattern validates formatting. If your project is INR-only, add this additional business rule:
- F.col("currency") != "INR"
- Check the product stock statuses. The allowed values in this notebook assume IN_STOCK, OUT_OF_STOCK, and LOW_STOCK. If your generated dataset uses different labels, adjust the allowed list to match the actual business definitions.
- Decimal types matter. Financial values should use DECIMAL, not floating-point types, to avoid unexpected rounding differences.
- This notebook reports failures but does not quarantine them. Quarantine handling is covered in a separate notebook.


## What is Quarantine in Data Engineering?


**Quarantine** is a `temporary storage area` where we isolate `bad, invalid, or suspicious records` instead of allowing them into the **clean Silver layer**.

In [0]:
# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

from datetime import datetime

print("Libraries imported successfully.")


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "data_type_format_report"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "quarantine_customers"
)

print("Project configuration completed.")


# ------------------------------------------------------------
# 3. DEFINE EXPECTED DATA TYPES
# ------------------------------------------------------------

EXPECTED_TYPES = {

    "silver_customers": {
        "customer_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "phone": "string",
        "city": "string",
        "state": "string",
        "created_at": "date"
    },

    "silver_products": {
        "product_id": "string",
        "product_name": "string",
        "category": "string",
        "unit_price": "decimal(12,2)",
        "unit_cost": "decimal(12,2)",
        "stock_status": "string",
        "created_at": "date"
    },

    "silver_orders": {
        "order_id": "string",
        "customer_id": "string",
        "order_timestamp": "timestamp",
        "order_status": "string",
        "currency": "string",
        "order_total": "decimal(12,2)",
        "sales_channel": "string"
    },

    "silver_order_items": {
        "order_item_id": "string",
        "order_id": "string",
        "product_id": "string",
        "quantity": "int",
        "unit_price": "decimal(12,2)",
        "discount_rate": "decimal(5,4)",
        "line_total": "decimal(12,2)"
    },

    "silver_payments": {
        "payment_id": "string",
        "order_id": "string",
        "payment_method": "string",
        "payment_status": "string",
        "amount": "decimal(12,2)",
        "payment_timestamp": "timestamp"
    },

    "silver_shipments": {
        "shipment_id": "string",
        "order_id": "string",
        "carrier": "string",
        "shipment_status": "string",
        "shipped_date": "timestamp",
        "delivered_date": "timestamp",
        "tracking_number": "string"
    },

    "silver_inventory": {
        "inventory_snapshot_id": "string",
        "product_id": "string",
        "warehouse_id": "string",
        "quantity_on_hand": "int",
        "snapshot_date": "date"
    }
}

print("Expected data types configured.")


# ------------------------------------------------------------
# 4. DEFINE ALLOWED CATEGORICAL VALUES
# ------------------------------------------------------------

ALLOWED_VALUES = {

    "silver_orders": {
        "order_status": [
            "DELIVERED",
            "SHIPPED",
            "PROCESSING",
            "CANCELLED"
        ]
    },

    "silver_products": {
        "stock_status": [
            "IN_STOCK",
            "OUT_OF_STOCK",
            "LOW_STOCK"
        ]
    },

    "silver_payments": {
        "payment_status": [
            "SUCCESS",
            "PENDING",
            "FAILED",
            "REFUNDED"
        ]
    },

    "silver_shipments": {
        "shipment_status": [
            "PENDING",
            "PROCESSING",
            "SHIPPED",
            "IN_TRANSIT",
            "OUT_FOR_DELIVERY",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        ]
    }
}

print("Allowed categorical values configured.")


# ------------------------------------------------------------
# 5. VALIDATION REPORT FUNCTIONS
# ------------------------------------------------------------

results = []


def add_result(
    dataset_name,
    check_name,
    column_name,
    total_records,
    failed_records,
    details
):

    total_records = int(total_records)
    failed_records = int(failed_records)

    failure_percentage = (
        round(
            failed_records / total_records * 100,
            2
        )
        if total_records > 0
        else 0.0
    )

    status = (
        "PASS"
        if total_records > 0 and failed_records == 0
        else "FAIL"
    )

    results.append({

        "dataset_name": dataset_name,

        "check_name": check_name,

        "column_name": column_name,

        "total_records": total_records,

        "failed_records": failed_records,

        "failure_percentage": float(failure_percentage),

        "status": status,

        "details": str(details),

        "checked_at": datetime.now()
    })


def count_failed_records(df, condition):

    return (
        df
        .filter(condition)
        .count()
    )


def get_silver_table(dataset_name):

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{dataset_name}"
    )

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Required Silver table does not exist: {table_name}"
        )

    return spark.table(table_name)


def validate_required_columns(
    df,
    dataset_name,
    required_columns
):

    available_columns = set(df.columns)

    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in available_columns
    ]

    if missing_columns:

        raise Exception(
            f"Missing columns in {dataset_name}: {missing_columns}"
        )


print("Validation functions created.")


# ------------------------------------------------------------
# 6. VALIDATE SPARK DATA TYPES
# ------------------------------------------------------------

for dataset_name, expected_columns in EXPECTED_TYPES.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    actual_types = {

        field.name: field.dataType.simpleString()

        for field in df.schema.fields
    }

    for column_name, expected_type in expected_columns.items():

        if column_name not in actual_types:

            add_result(
                dataset_name,
                "data_type_validation",
                column_name,
                total_records,
                1,
                f"Missing column: {column_name}"
            )

            continue

        actual_type = actual_types[column_name]

        failed = (
            0
            if actual_type == expected_type
            else 1
        )

        add_result(
            dataset_name,
            "data_type_validation",
            column_name,
            total_records,
            failed,
            (
                f"Expected: {expected_type}; "
                f"Actual: {actual_type}"
            )
        )

print("Data type validation completed.")


# ------------------------------------------------------------
# 7. VALIDATE CUSTOMER EMAIL FORMAT
# ------------------------------------------------------------

customers_df = get_silver_table(
    "silver_customers"
)

validate_required_columns(
    customers_df,
    "silver_customers",
    ["email"]
)

email_pattern = (
    r"^[A-Za-z0-9._%+-]+@"
    r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

invalid_email_condition = (

    F.col("email").isNotNull()

    & (
        ~F.trim(F.col("email")).rlike(email_pattern)
    )
)

invalid_email_count = count_failed_records(
    customers_df,
    invalid_email_condition
)

add_result(
    "silver_customers",
    "email_format_validation",
    "email",
    customers_df.count(),
    invalid_email_count,
    "Non-null email values must match the configured email pattern."
)

print("Email format validation completed.")


# ------------------------------------------------------------
# 8. VALIDATE CUSTOMER PHONE FORMAT
# ------------------------------------------------------------

# Accepted:
# 9876543210
# +919876543210
# 98765-43210
# (98765) 43210
#
# Formatting characters are removed before validation.


phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)

invalid_phone_condition = (

    F.col("phone").isNotNull()

    & (
        ~normalized_phone.rlike(phone_pattern)
    )
)

invalid_phone_count = count_failed_records(
    customers_df,
    invalid_phone_condition
)

add_result(
    "silver_customers",
    "phone_format_validation",
    "phone",
    customers_df.count(),
    invalid_phone_count,
    "Non-null phone values must contain 10-15 digits, optionally prefixed with +."
)

print("Phone format validation completed.")


# ------------------------------------------------------------
# 9. VALIDATE CATEGORICAL VALUES
# ------------------------------------------------------------

for dataset_name, columns in ALLOWED_VALUES.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    for column_name, allowed_values in columns.items():

        validate_required_columns(
            df,
            dataset_name,
            [column_name]
        )

        invalid_condition = (

            F.col(column_name).isNull()

            | (
                ~F.col(column_name).isin(allowed_values)
            )
        )

        invalid_count = count_failed_records(
            df,
            invalid_condition
        )

        add_result(
            dataset_name,
            "allowed_category_validation",
            column_name,
            total_records,
            invalid_count,
            f"Allowed values: {allowed_values}"
        )

print("Categorical validation completed.")


# ------------------------------------------------------------
# 10. VALIDATE CURRENCY CODES
# ------------------------------------------------------------

orders_df = get_silver_table(
    "silver_orders"
)

validate_required_columns(
    orders_df,
    "silver_orders",
    ["currency"]
)

invalid_currency_condition = (

    F.col("currency").isNull()

    | (
        ~F.trim(F.col("currency")).rlike(r"^[A-Z]{3}$")
    )
)

invalid_currency_count = count_failed_records(
    orders_df,
    invalid_currency_condition
)

add_result(
    "silver_orders",
    "currency_format_validation",
    "currency",
    orders_df.count(),
    invalid_currency_count,
    "Currency must be a three-letter uppercase code."
)

print("Currency validation completed.")


# ------------------------------------------------------------
# 11. VALIDATE NUMERIC BUSINESS RULES
# ------------------------------------------------------------

# Store rule names and column names.
# Build each Spark condition inside the loop.


numeric_rules = {

    "silver_products": [

        (
            "unit_price",
            lambda: F.col("unit_price") < 0
        ),

        (
            "unit_cost",
            lambda: F.col("unit_cost") < 0
        )
    ],

    "silver_orders": [

        (
            "order_total",
            lambda: F.col("order_total") < 0
        )
    ],

    "silver_order_items": [

        (
            "quantity",
            lambda: F.col("quantity") <= 0
        ),

        (
            "unit_price",
            lambda: F.col("unit_price") < 0
        ),

        (
            "discount_rate",
            lambda: (
                (F.col("discount_rate") < 0)
                | (F.col("discount_rate") > 1)
            )
        ),

        (
            "line_total",
            lambda: F.col("line_total") < 0
        )
    ],

    "silver_payments": [

        (
            "amount",
            lambda: F.col("amount") < 0
        )
    ],

    "silver_inventory": [

        (
            "quantity_on_hand",
            lambda: F.col("quantity_on_hand") < 0
        )
    ]
}


for dataset_name, rules in numeric_rules.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    for column_name, condition_function in rules:

        validate_required_columns(
            df,
            dataset_name,
            [column_name]
        )

        invalid_condition = condition_function()

        failed_count = count_failed_records(
            df,
            invalid_condition
        )

        add_result(
            dataset_name,
            "numeric_business_rule",
            column_name,
            total_records,
            failed_count,
            f"Invalid numeric values found in {column_name}."
        )

print("Numeric business rule validation completed.")


# ------------------------------------------------------------
# 12. VALIDATE ORDER ITEM ARITHMETIC
# ------------------------------------------------------------

# Expected:
# quantity * unit_price * (1 - discount_rate)
#
# Allowed rounding tolerance: 0.02


order_items_df = get_silver_table(
    "silver_order_items"
)

validate_required_columns(
    order_items_df,
    "silver_order_items",
    [
        "quantity",
        "unit_price",
        "discount_rate",
        "line_total"
    ]
)


expected_line_total = (

    F.col("quantity").cast("decimal(18,4)")

    * F.col("unit_price").cast("decimal(18,4)")

    * (
        F.lit(1).cast("decimal(18,4)")

        - F.col("discount_rate").cast("decimal(18,4)")
    )
)


difference = (

    F.col("line_total").cast("decimal(18,4)")

    - expected_line_total
)


invalid_line_total_condition = (

    F.col("quantity").isNotNull()

    & F.col("unit_price").isNotNull()

    & F.col("discount_rate").isNotNull()

    & F.col("line_total").isNotNull()

    & (
        (difference > F.lit(0.02))
        | (difference < F.lit(-0.02))
    )
)


invalid_line_total_count = count_failed_records(
    order_items_df,
    invalid_line_total_condition
)

add_result(
    "silver_order_items",
    "line_total_reconciliation",
    "line_total",
    order_items_df.count(),
    invalid_line_total_count,
    "Line total must match quantity * unit price * (1 - discount rate), within 0.02."
)

print("Order item arithmetic validation completed.")


# ------------------------------------------------------------
# 13. VALIDATE SHIPMENT DATE CONSISTENCY
# ------------------------------------------------------------

shipments_df = get_silver_table(
    "silver_shipments"
)

validate_required_columns(
    shipments_df,
    "silver_shipments",
    [
        "shipment_status",
        "shipped_date",
        "delivered_date"
    ]
)


# Rule 1:
# Delivered shipments require delivered_date.

missing_delivery_date = (

    (F.col("shipment_status") == "DELIVERED")

    & F.col("delivered_date").isNull()
)


# Rule 2:
# Delivery date cannot exist without shipment date.

delivery_without_shipment_date = (

    F.col("delivered_date").isNotNull()

    & F.col("shipped_date").isNull()
)


# Rule 3:
# Delivery cannot happen before shipment.

delivery_before_shipment = (

    F.col("delivered_date").isNotNull()

    & F.col("shipped_date").isNotNull()

    & (
        F.col("delivered_date") < F.col("shipped_date")
    )
)


invalid_shipment_dates = (

    missing_delivery_date

    | delivery_without_shipment_date

    | delivery_before_shipment
)


invalid_shipment_date_count = count_failed_records(
    shipments_df,
    invalid_shipment_dates
)

add_result(
    "silver_shipments",
    "shipment_date_consistency",
    "delivered_date",
    shipments_df.count(),
    invalid_shipment_date_count,
    "Delivered shipments require a delivery date. Delivery cannot exist without shipment date or precede shipment."
)

print("Shipment date validation completed.")


# ------------------------------------------------------------
# 14. VALIDATE CUSTOMER QUARANTINE TABLE
# ------------------------------------------------------------

if not spark.catalog.tableExists(QUARANTINE_TABLE):

    raise Exception(
        f"Customer quarantine table not found: {QUARANTINE_TABLE}"
    )


quarantine_df = spark.table(
    QUARANTINE_TABLE
)


validate_required_columns(
    quarantine_df,
    "quarantine_customers",
    [
        "customer_id",
        "phone",
        "quarantine_reason",
        "quarantined_at"
    ]
)


quarantine_count = quarantine_df.count()


print(
    "Customer quarantine records:",
    quarantine_count
)


# Check 1:
# Every quarantine record must have a reason.

missing_quarantine_reason = (

    F.col("quarantine_reason").isNull()

    | (
        F.trim(F.col("quarantine_reason")) == ""
    )
)


missing_reason_count = count_failed_records(
    quarantine_df,
    missing_quarantine_reason
)


add_result(
    "quarantine_customers",
    "quarantine_reason_validation",
    "quarantine_reason",
    quarantine_count,
    missing_reason_count,
    "Every quarantined customer must have a non-empty quarantine reason."
)


# Check 2:
# Quarantined records must contain invalid phone values.
#
# This counts records that unexpectedly have valid phone numbers.


quarantine_normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)


valid_phone_in_quarantine = (

    F.col("phone").isNull()

    | quarantine_normalized_phone.rlike(phone_pattern)
)


valid_phone_quarantine_count = count_failed_records(
    quarantine_df,
    valid_phone_in_quarantine
)


add_result(
    "quarantine_customers",
    "quarantine_phone_validation",
    "phone",
    quarantine_count,
    valid_phone_quarantine_count,
    "Quarantined customer records must have invalid phone values."
)


# Check 3:
# Customer IDs must be unique in quarantine.


duplicate_quarantine_ids = (

    quarantine_df

    .groupBy("customer_id")

    .count()

    .filter(F.col("count") > 1)

    .count()
)


add_result(
    "quarantine_customers",
    "quarantine_duplicate_validation",
    "customer_id",
    quarantine_count,
    duplicate_quarantine_ids,
    "Customer IDs must not be duplicated in the quarantine table."
)


print("Customer quarantine validation completed.")


# ------------------------------------------------------------
# 15. VERIFY INVALID PHONES ARE NOT IN SILVER
# ------------------------------------------------------------

invalid_phone_in_silver_count = count_failed_records(
    customers_df,
    invalid_phone_condition
)


add_result(
    "silver_customers",
    "invalid_phone_quarantine_verification",
    "phone",
    customers_df.count(),
    invalid_phone_in_silver_count,
    "Invalid phone numbers must be quarantined and must not remain in Silver customers."
)


print("Silver and quarantine consistency validation completed.")


# ------------------------------------------------------------
# 16. CREATE VALIDATION REPORT
# ------------------------------------------------------------

report_schema = StructType([

    StructField(
        "dataset_name",
        StringType(),
        False
    ),

    StructField(
        "check_name",
        StringType(),
        False
    ),

    StructField(
        "column_name",
        StringType(),
        True
    ),

    StructField(
        "total_records",
        LongType(),
        False
    ),

    StructField(
        "failed_records",
        LongType(),
        False
    ),

    StructField(
        "failure_percentage",
        DoubleType(),
        False
    ),

    StructField(
        "status",
        StringType(),
        False
    ),

    StructField(
        "details",
        StringType(),
        True
    ),

    StructField(
        "checked_at",
        TimestampType(),
        False
    )
])


if len(results) == 0:

    raise Exception(
        "No validation results were generated."
    )


report_df = spark.createDataFrame(
    results,
    schema=report_schema
)


print("Validation report created.")


display(
    report_df.orderBy(
        "dataset_name",
        "check_name",
        "column_name"
    )
)


# ------------------------------------------------------------
# 17. SAVE REPORT TO DELTA TABLE
# ------------------------------------------------------------

(
    report_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(REPORT_TABLE)
)


print(
    f"Report saved successfully: {REPORT_TABLE}"
)


# ------------------------------------------------------------
# 18. DATASET-LEVEL VALIDATION SUMMARY
# ------------------------------------------------------------

summary_df = (

    report_df

    .groupBy("dataset_name")

    .agg(

        F.count("*").alias("total_checks"),

        F.sum(

            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)

        ).alias("passed_checks"),

        F.sum(

            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)

        ).alias("failed_checks")
    )

    .withColumn(

        "overall_status",

        F.when(
            F.col("failed_checks") == 0,
            "PASS"
        ).otherwise("FAIL")
    )

    .orderBy("dataset_name")
)


display(summary_df)


# ------------------------------------------------------------
# 19. FINAL VALIDATION SUMMARY
# ------------------------------------------------------------

total_checks = report_df.count()


passed_checks = (

    report_df

    .filter(
        F.col("status") == "PASS"
    )

    .count()
)


failed_checks = (

    report_df

    .filter(
        F.col("status") == "FAIL"
    )

    .count()
)


print("=" * 65)

print("DATA TYPE, FORMAT AND QUARANTINE VALIDATION SUMMARY")

print("=" * 65)

print(f"Total checks  : {total_checks}")

print(f"Passed checks : {passed_checks}")

print(f"Failed checks : {failed_checks}")

print("=" * 65)


if failed_checks == 0:

    print("RESULT: ALL VALIDATION CHECKS PASSED.")

else:

    print(
        f"RESULT: {failed_checks} VALIDATION CHECK(S) FAILED."
    )


print()
print("CUSTOMER RECORD SUMMARY")

print(
    "Silver customers:",
    customers_df.count()
)

print(
    "Quarantined customers:",
    quarantine_count
)

print()
print("REPORT TABLE")

print(REPORT_TABLE)


# ------------------------------------------------------------
# 20. DISPLAY FINAL SAVED REPORT
# ------------------------------------------------------------

display(

    spark.table(REPORT_TABLE)

    .orderBy(
        "dataset_name",
        "check_name",
        "column_name"
    )
)


print("Data type, format and quarantine validation completed.")


In [0]:
# Check Bronze tables
spark.sql("""
SHOW TABLES IN workspace.ecommerce_bronze
""").show(truncate=False)

# Check quarantine table schema
spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
).printSchema()

# Check quarantine table contents
spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
).show(10, truncate=False)


### Recover and append invalid phone records

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. TABLE CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.ecommerce_bronze.bronze_customers"
SILVER_TABLE = "workspace.ecommerce_silver.silver_customers"
QUARANTINE_TABLE = "workspace.ecommerce_silver.quarantine_customers"


# ============================================================
# 2. READ BRONZE AND QUARANTINE TABLES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE)

quarantine_df = spark.table(QUARANTINE_TABLE)

bronze_columns = bronze_df.columns

print("Bronze records:", bronze_df.count())
print("Existing quarantine records:", quarantine_df.count())


# ============================================================
# 3. IDENTIFY INVALID PHONE NUMBERS
# ============================================================

# Remove spaces, brackets and hyphens before validating.
normalized_phone = F.regexp_replace(
    F.col("phone"),
    r"[\s()\-]",
    ""
)

# Valid format:
# 10-15 digits, optionally starting with +

invalid_phone_condition = (
    F.col("phone").isNotNull()
    & ~normalized_phone.rlike(r"^\+?[0-9]{10,15}$")
)

invalid_records = bronze_df.filter(invalid_phone_condition)

print("Invalid phone records found in Bronze:", invalid_records.count())

display(
    invalid_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone"
    )
)


# ============================================================
# 4. REMOVE RECORDS ALREADY PRESENT IN QUARANTINE
# ============================================================

existing_quarantine_ids = (
    quarantine_df
    .select("customer_id")
    .filter(F.col("customer_id").isNotNull())
    .distinct()
)

new_records = invalid_records.join(
    existing_quarantine_ids,
    on="customer_id",
    how="left_anti"
)

print("New records to quarantine:", new_records.count())


# ============================================================
# 5. ADD QUARANTINE INFORMATION
# ============================================================

new_records = (
    new_records
    .withColumn(
        "quarantine_reason",
        F.lit("INVALID_PHONE_FORMAT")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)


# ============================================================
# 6. ALIGN DATAFRAME WITH EXISTING TABLE SCHEMA
# ============================================================

target_schema = quarantine_df.schema

aligned_expressions = []

for field in target_schema.fields:

    column_name = field.name

    # Map Bronze ingestion timestamp to quarantine timestamp.
    if column_name == "_ingestion_timestamp":

        if "_ingested_at" in bronze_columns:
            expression = F.col("_ingested_at").cast(field.dataType)
        elif "_ingestion_timestamp" in bronze_columns:
            expression = F.col("_ingestion_timestamp").cast(field.dataType)
        else:
            expression = F.lit(None).cast(field.dataType)

    # Derive ingestion date if necessary.
    elif column_name == "_ingestion_date":

        if "_ingestion_date" in bronze_columns:
            expression = F.col("_ingestion_date").cast(field.dataType)
        elif "_ingested_at" in bronze_columns:
            expression = F.to_date("_ingested_at").cast(field.dataType)
        elif "_ingestion_timestamp" in bronze_columns:
            expression = F.to_date("_ingestion_timestamp").cast(field.dataType)
        else:
            expression = F.lit(None).cast(field.dataType)

    # Preserve source columns that exist.
    elif column_name in new_records.columns:

        expression = F.col(column_name).cast(field.dataType)

    # Fill missing target columns with typed NULL.
    else:

        expression = F.lit(None).cast(field.dataType)

    aligned_expressions.append(
        expression.alias(column_name)
    )


aligned_records = new_records.select(*aligned_expressions)


# ============================================================
# 7. APPEND TO QUARANTINE TABLE
# ============================================================

records_to_insert = aligned_records.count()

if records_to_insert > 0:

    (
        aligned_records.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

    print("Quarantine append completed.")

else:

    print("No new invalid phone records to append.")


# ============================================================
# 8. VERIFY QUARANTINE TABLE
# ============================================================

updated_quarantine = spark.table(QUARANTINE_TABLE)

print("Total quarantined customer records:",
      updated_quarantine.count())

display(
    updated_quarantine.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone",
        "quarantine_reason",
        "quarantined_at"
    )
)


### Verify the three quarantine conditions:


After the append succeeds, run:

In [0]:
from pyspark.sql import functions as F

quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
)

silver_df = spark.table(
    "workspace.ecommerce_silver.silver_customers"
)

# 1. Check duplicate customer IDs
duplicate_ids = (
    quarantine_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate quarantine IDs:", duplicate_ids.count())


# 2. Check invalid phone numbers
normalized_phone = F.regexp_replace(
    F.col("phone"),
    r"[\s()\-]",
    ""
)

valid_phone_condition = normalized_phone.rlike(
    r"^\+?[0-9]{10,15}$"
)

valid_quarantine_phones = quarantine_df.filter(
    F.col("phone").isNull() | valid_phone_condition
)

print("Invalid quarantine phone records:",
      quarantine_df.count() - valid_quarantine_phones.count())


# 3. Check quarantine reasons
missing_reasons = quarantine_df.filter(
    F.col("quarantine_reason").isNull()
    | (F.trim(F.col("quarantine_reason")) == "")
)

print("Records with missing quarantine reasons:",
      missing_reasons.count())


# 4. Check invalid phones remaining in Silver
silver_invalid_phones = silver_df.filter(
    F.col("phone").isNotNull()
    & ~F.regexp_replace(
        F.col("phone"),
        r"[\s()\-]",
        ""
    ).rlike(r"^\+?[0-9]{10,15}$")
)

print("Invalid phone records remaining in Silver:",
      silver_invalid_phones.count())


# 5. Final counts
print("Silver customers:", silver_df.count())
print("Quarantined customers:", quarantine_df.count())


### Move invalid customers into quarantine:

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = (
    "workspace.ecommerce_silver.silver_customers"
)

QUARANTINE_TABLE = (
    "workspace.ecommerce_silver.quarantine_customers"
)

customers_df = spark.table(SILVER_TABLE)

# Identify invalid phone numbers
phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)

invalid_condition = (
    F.col("phone").isNotNull()
    & (~normalized_phone.rlike(phone_pattern))
)

# Separate valid and invalid records
valid_customers = customers_df.filter(
    ~invalid_condition
)

quarantined_customers = (
    customers_df
    .filter(invalid_condition)
    .withColumn(
        "quarantine_reason",
        F.lit("Invalid phone number format")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

# Display counts before writing
print("Original customers:", customers_df.count())
print("Valid customers:", valid_customers.count())
print("Quarantined customers:", quarantined_customers.count())

# Save valid customers
(
    valid_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# Save quarantined customers
(
    quarantined_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Customer phone quarantine completed.")


### Verify quarantine records:

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "workspace.ecommerce_silver.silver_customers"
QUARANTINE_TABLE = "workspace.ecommerce_silver.quarantine_customers"

silver_df = spark.table(SILVER_TABLE)

print("Silver customer count:", silver_df.count())

print("Quarantine table exists:",
      spark.catalog.tableExists(QUARANTINE_TABLE))

if spark.catalog.tableExists(QUARANTINE_TABLE):

    quarantine_df = spark.table(QUARANTINE_TABLE)

    print("Quarantine record count:", quarantine_df.count())

    display(quarantine_df)


### Check whether the invalid records are still in Silver:

In [0]:
phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)

invalid_condition = (
    F.col("phone").isNotNull()
    & (~normalized_phone.rlike(phone_pattern))
)

invalid_records = silver_df.filter(invalid_condition)

print("Invalid phone records still in Silver:",
      invalid_records.count())

display(
    invalid_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone"
    )
)


### Recover the three records from Bronze:

In [0]:
from pyspark.sql import functions as F

BRONZE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse/bronze/customers"
)

QUARANTINE_TABLE = (
    "workspace.ecommerce_silver.quarantine_customers"
)

# Read original Bronze data
bronze_customers = (
    spark.read
    .format("delta")
    .load(BRONZE_PATH)
)

# Identify the three affected customers
customer_ids = [
    "CUST000212",
    "CUST003211",
    "CUST008801"
]

recovered_records = (
    bronze_customers
    .filter(F.col("customer_id").isin(customer_ids))
    .withColumn(
        "quarantine_reason",
        F.lit("Invalid phone number format")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

print("Recovered records:", recovered_records.count())

display(
    recovered_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone",
        "quarantine_reason"
    )
)